# 04 — Gold Business Metrics

**Project:** Orchestrated Incremental Orders Pipeline with Databricks Workflows  
**Layer:** Gold  
**Source:** `orders_workflow_project.silver_orders_current`  
**Targets:**

- `orders_workflow_project.gold_orders_by_status`
- `orders_workflow_project.gold_daily_sales`
- `orders_workflow_project.gold_order_summary`

This notebook creates business-ready Gold tables from the Silver current-state orders table.

The notebook is designed to be executed by a Databricks Workflow / Job using the parameter:

- `pipeline_run_id`

The Gold layer answers business questions such as:

- How many current orders exist by status?
- How much revenue has been recognized?
- What is the cancellation rate?
- What is the delivered rate?
- What is the average order value?
- What is the total active order amount?

In [0]:
dbutils.widgets.text("pipeline_run_id", "manual_run", "Pipeline Run ID")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print(f"Pipeline Run ID: {pipeline_run_id}")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
spark.sql("USE SCHEMA orders_workflow_project")

In [0]:
silver_table = "orders_workflow_project.silver_orders_current"

gold_orders_by_status_table = "orders_workflow_project.gold_orders_by_status"
gold_daily_sales_table = "orders_workflow_project.gold_daily_sales"
gold_order_summary_table = "orders_workflow_project.gold_order_summary"

audit_table = "orders_workflow_project.pipeline_run_audit"

print(f"Silver table: {silver_table}")
print(f"Gold orders by status table: {gold_orders_by_status_table}")
print(f"Gold daily sales table: {gold_daily_sales_table}")
print(f"Gold order summary table: {gold_order_summary_table}")
print(f"Audit table: {audit_table}")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

print(f"Silver table exists: {silver_exists}")

if not silver_exists:
    raise Exception(
        "Silver table does not exist. "
        "Run the Silver MERGE notebook before running Gold metrics."
    )

In [0]:
silver_df = spark.table(silver_table)

silver_count = silver_df.count()

print(f"Silver current orders count: {silver_count}")

display(
    silver_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_amount",
        "last_event_ts",
        "last_batch_id"
    ).orderBy("order_id")
)

In [0]:
if silver_count == 0:
    raise Exception(
        "Silver table exists but contains zero records. "
        "Gold metrics cannot be created from an empty Silver table."
    )

In [0]:
silver_df.createOrReplaceTempView("silver_orders_current_temp")

print("Temporary view created: silver_orders_current_temp")

In [0]:
gold_orders_by_status_df = spark.sql("""
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_amount_usd,
    ROUND(AVG(order_amount), 2) AS avg_order_amount_usd,
    MIN(last_event_ts) AS first_event_ts,
    MAX(last_event_ts) AS latest_event_ts,
    current_timestamp() AS gold_processed_ts
FROM silver_orders_current_temp
GROUP BY order_status
ORDER BY total_orders DESC, order_status ASC
""")

display(gold_orders_by_status_df)

In [0]:
gold_orders_by_status_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_orders_by_status_table)

print(f"Gold table created: {gold_orders_by_status_table}")

In [0]:
gold_daily_sales_df = spark.sql("""
SELECT
    TO_DATE(last_event_ts) AS order_activity_date,

    COUNT(*) AS total_current_orders,

    ROUND(SUM(order_amount), 2) AS gross_current_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'delivered' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS recognized_revenue_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS cancelled_amount_usd,

    SUM(CASE WHEN order_status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
    SUM(CASE WHEN order_status = 'shipped' THEN 1 ELSE 0 END) AS shipped_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,

    current_timestamp() AS gold_processed_ts

FROM silver_orders_current_temp
GROUP BY TO_DATE(last_event_ts)
ORDER BY order_activity_date
""")

display(gold_daily_sales_df)

In [0]:
gold_daily_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_daily_sales_table)

print(f"Gold table created: {gold_daily_sales_table}")

In [0]:
gold_order_summary_df = spark.sql("""
SELECT
    COUNT(*) AS total_current_orders,
    COUNT(DISTINCT customer_id) AS total_customers,

    ROUND(SUM(order_amount), 2) AS gross_current_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'delivered' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS recognized_revenue_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status != 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS active_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS cancelled_amount_usd,

    SUM(CASE WHEN order_status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
    SUM(CASE WHEN order_status = 'shipped' THEN 1 ELSE 0 END) AS shipped_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,

    ROUND(AVG(order_amount), 2) AS avg_order_value_usd,

    ROUND(
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) / COUNT(*) * 100,
        2
    ) AS delivered_rate_pct,

    ROUND(
        SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*) * 100,
        2
    ) AS cancellation_rate_pct,

    current_timestamp() AS gold_processed_ts

FROM silver_orders_current_temp
""")

display(gold_order_summary_df)

In [0]:
gold_order_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_order_summary_table)

print(f"Gold table created: {gold_order_summary_table}")

In [0]:
display(
    spark.sql("""
    SHOW TABLES IN orders_workflow_project
    """)
)

In [0]:
gold_validation_counts_df = spark.sql("""
SELECT 'bronze_orders_raw' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.bronze_orders_raw

UNION ALL

SELECT 'silver_orders_current' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.silver_orders_current

UNION ALL

SELECT 'gold_orders_by_status' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.gold_orders_by_status

UNION ALL

SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.gold_daily_sales

UNION ALL

SELECT 'gold_order_summary' AS table_name, COUNT(*) AS total_records
FROM orders_workflow_project.gold_order_summary
""")

display(gold_validation_counts_df)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.gold_orders_by_status
    ORDER BY total_orders DESC, order_status ASC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.gold_daily_sales
    ORDER BY order_activity_date
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.gold_order_summary
    """)
)

In [0]:
gold_records_processed = (
    gold_orders_by_status_df.count()
    + gold_daily_sales_df.count()
    + gold_order_summary_df.count()
)

print(f"Gold records processed: {gold_records_processed}")

In [0]:
pipeline_run_id_sql = pipeline_run_id.replace("'", "''")
task_name_sql = "04_gold_business_metrics"
status_sql = "SUCCESS"
message_sql = "Gold business metrics created successfully."

spark.sql(f"""
INSERT INTO {audit_table}
SELECT
    '{pipeline_run_id_sql}' AS pipeline_run_id,
    '{task_name_sql}' AS task_name,
    '{status_sql}' AS status,
    CAST({int(gold_records_processed)} AS BIGINT) AS records_processed,
    '{message_sql}' AS message,
    current_timestamp() AS processed_ts
""")

print("Audit record inserted successfully.")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.pipeline_run_audit
    ORDER BY processed_ts DESC
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY orders_workflow_project.gold_order_summary
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)